In [3]:
import requests
import json
import base64
from PIL import Image
import io
import numpy as np
from dotenv import load_dotenv
import os
import gradio as gr

# Load API key từ file .env
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# 4. Hàm xử lý câu hỏi văn bản (Gemini)
def ask_gemini(text):
    if not GEMINI_API_KEY:
        return "⚠️ Không tìm thấy API key cho Gemini."

    from genai import GenerateModel
    model = GenerateModel(model="gemini-2-0-flash", api_key=GEMINI_API_KEY)
    response = model.generate_content(text)
    return response.text.strip()

# 5. Hàm xử lý hình ảnh và trả lời câu hỏi (Ollama)
def describe_and_answer(image_np):
    if not isinstance(image_np, np.ndarray):
        return "⚠️ Đầu vào không phải là hình ảnh."

    img = Image.fromarray(image_np)
    img_buffer = io.BytesIO()
    img.save(img_buffer, format='JPEG')
    img_bytes = img_buffer.getvalue()

    img_base64 = base64.b64encode(img_bytes).decode('utf-8')

    payload = {
        "model": "llava:7b",
        "messages": [{"role": "user", "content": "Mô tả bức ảnh này với các chi tiết rõ ràng, tập trung vào giày và tính năng của chúng. Giày này có phù hợp để đá bóng không?"}],
        "images": [img_base64],
        "stream": True
    }

    try:
        response = requests.post("http://localhost:11434/api/chat", json=payload, stream=True)
        response.raise_for_status()

        content = ""
        for line in response.iter_lines():
            if line:
                part = json.loads(line.decode("utf-8"))
                content += part.get("message", {}).get("content", "")

        if "giày" not in content.lower():
            return "⚠️ Mô tả không liên quan đến nội dung ảnh. Vui lòng thử lại."

        return content.strip() if content else "⚠️ Không nhận được phản hồi từ mô hình Ollama."

    except Exception as e:
        return f"⚠️ Lỗi khi gọi Ollama: {e}"

# 6. (Nâng cao) Hàm agent giả lập tìm kiếm Google
def agent_search(query):
    return f"[Agent] Tôi đã tìm kiếm Google và thấy thông tin: {query} - Kết quả mô phỏng."

# 7. Hàm chính xử lý input
def chatbot_response(message, image=None):
    if image is not None:
        description = describe_and_answer(image)
        return f"🖼️ Phân tích và trả lời: {description}"
    elif "tìm kiếm" in message.lower():
        search_result = agent_search(message)
        return search_result
    else:
        answer = ask_gemini(message)
        return answer

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## Multi-Modal Chatbot Demo")
    text_input = gr.Textbox(label="Nhập câu hỏi (hoặc yêu cầu tìm kiếm)")
    image_input = gr.Image(label="Tải lên ảnh (tùy chọn)", type="numpy")
    submit = gr.Button("Gửi")
    output = gr.Textbox(label="Phản hồi")

    submit.click(fn=chatbot_response, inputs=[text_input, image_input], outputs=output)

demo.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
